# import

In [1]:
import sys
sys.path.append('..')
import pandas as pd
import numpy as np
import cv2 as cv
from pathlib import Path 
import seaborn as sns
import matplotlib.pyplot as plt
from StatTools.generators.ndfnoise_generator import ndfnoise
from tqdm import tqdm
import plotly.express as px
import os
import warnings 
import gc
from synth import calc_soi_counts, calc_soi_area, draw_traj

# frufa

In [ ]:
result = []
frame_shape = (800, 1300)
thikness = 10
dir_path = Path(fr'../data/knn_kalman/frufa')
files = list(dir_path.glob("*.csv"))
pbar = tqdm(files)
for file_path in pbar:
    pbar.set_description(f"Processing {file_path.name}")
    df_test = pd.read_csv(file_path)[['frame', 'track_id', 'x', 'y']]
    df_test = df_test.astype({'x':np.int32, 'y':np.int32})
    
    x_wide = df_test.pivot_table(columns='track_id', index='frame', values='x', fill_value=np.nan)
    y_wide = df_test.pivot_table(columns='track_id', index='frame', values='y', fill_value=np.nan)
    trajectories = np.stack([x_wide.values, y_wide.values], axis=-1)
    del df_test
    del x_wide
    del y_wide
    gc.collect()
    frame_thresholds = np.exp(np.arange(np.log(10), np.log(trajectories.shape[0]), 0.5)).astype(np.int32)
    
    try:
        for frame in frame_thresholds:
            # track_slice = trajectories[:frame]
            iou, _, _  = calc_iou(trajectories[:frame], thikness, frame_shape)

            diff = np.diff(trajectories[:frame], axis=0)

            total_l1_path = np.nansum(np.abs(diff))
            total_l2_path = np.nansum(np.linalg.norm(diff, axis=2))
            del diff
            result.append({'species': 'F.rufa',
                            'name': file_path.name,
                            'frame_threshold':frame,
                            'total_l1_path':total_l1_path,
                            'total_l2_path':total_l2_path,
                            'iou': iou,
                        })
    except Exception as e:
        warnings.warn(f"Ошибка в {file_path.name}: {e}")
        continue
df_res = pd.DataFrame(result)
df_res.to_pickle('../data/artifacts/knn_kalman_frufa_soi.pkl')

# pyeensis

In [ ]:

frame_shape = (800, 1300)
thikness = 10
dir_path = Path(fr'../data/knn_kalman/pyeensis')
files = list(dir_path.glob("*.csv"))
pbar = tqdm(files)
for file_path in pbar:
    result = []
    pbar.set_description(f"Processing {file_path.name}")
    df_test = pd.read_csv(file_path)[['frame', 'track_id', 'x', 'y']]
    df_test = df_test.astype({'x':np.int32, 'y':np.int32})
    
    x_wide = df_test.pivot_table(columns='track_id', index='frame', values='x', fill_value=np.nan)
    y_wide = df_test.pivot_table(columns='track_id', index='frame', values='y', fill_value=np.nan)
    trajectories = np.stack([x_wide.values, y_wide.values], axis=-1)
    del df_test
    del x_wide
    del y_wide
    gc.collect()
    frame_thresholds = np.exp(np.arange(np.log(10), np.log(trajectories.shape[0]), 0.5)).astype(np.int32)
    
    try:
        for frame in frame_thresholds:
            # track_slice = trajectories[:frame]
            soi_area, _, _  = calc_soi_area(trajectories[:frame], thikness, frame_shape)

            diff = np.diff(trajectories[:frame], axis=0)

            total_l1_path = np.nansum(np.abs(diff))
            total_l2_path = np.nansum(np.linalg.norm(diff, axis=2))
            del diff
            result.append({'species': 'P.yeensis',
                            'name': file_path.name,
                            'frame_threshold':frame,
                            'total_l1_path':total_l1_path,
                            'total_l2_path':total_l2_path,
                            'soi_area': soi_area,
                        })
    except Exception as e:
        warnings.warn(f"Ошибка в {file_path.name}: {e}")
        continue
    df_res = pd.DataFrame(result)
    df_res.to_pickle(f'../data/artifacts/knn_kalman_pyeensis_soi/{file_path.name}.pkl')

# debug

## draw 3d long tracks

In [ ]:
# df_test = pd.read_csv('../data/knn_kalman/pyeensis/S2190011.csv')[['frame', 'track_id', 'x', 'y']]
# df_test = df_test.round().astype({'x':np.int32, 'y':np.int16})
# track_lens = df_test.groupby('track_id')['frame'].count()
# long_enough_tracks = track_lens[track_lens>1000].index.to_list()
# df_test = df_test[df_test.track_id.isin(long_enough_tracks)]
# # df_sparse = df_test.groupby('track_id', as_index=True).apply(lambda x: x.iloc[::1])
# # df_sparse = df_sparse.reset_index().drop(columns='level_1')
# fig = px.line_3d(df_test, x='x', y='y', z='frame', 
#                   color='track_id',
#                 #   hover_data=['frame'],
#                   )
# fig.update_traces(marker=dict(size=1))
# fig.write_html('../data/artifacts/3d_pyeensis_tracks.html')


In [ ]:
# df_test = pd.read_csv('../data/knn_kalman/frufa/S1240015.csv')[['frame', 'track_id', 'x', 'y']]
# df_test = df_test.round().astype({'x':np.int32, 'y':np.int16})
# track_lens = df_test.groupby('track_id')['frame'].count()
# long_enough_tracks = track_lens[track_lens>500].index.to_list()
# df_test = df_test[df_test.track_id.isin(long_enough_tracks)]
# # df_sparse = df_test.groupby('track_id', as_index=True).apply(lambda x: x.iloc[::1])
# # df_sparse = df_sparse.reset_index().drop(columns='level_1')
# fig = px.line_3d(df_test, x='x', y='y', z='frame', 
#                   color='track_id',
#                 #   hover_data=['frame'],
#                   )
# fig.update_traces(marker=dict(size=1))
# fig.write_html('../data/artifacts/3d_frufa_tracks.html')


## debug prefix sum

In [2]:
import numpy as np
import cv2 as cv

def calc_soi_area_incremental(trajectories_chunk, thickness, frame_shape, coverage_mask):
    """
    Инкрементальный расчёт площади SOI.
    
    Parameters:
        trajectories_chunk: np.ndarray shape (N_frames, N_tracks, 2)
                            Срез траекторий ТОЛЬКО для новых кадров [start:end].
        thickness: int      Толщина линии для рисования.
        frame_shape: tuple  (height, width) кадра.
        coverage_mask: np.ndarray shape (H, W), dtype=np.uint8
                       Внешняя маска, которая накапливает покрытие.
                       Перед первым вызовом должна быть создана как np.zeros(..., dtype=np.uint8).
                       
    Returns:
        float: Текущая площадь SOI (количество ненулевых пикселей в coverage_mask).
    """
    n_tracks = trajectories_chunk.shape[1]
    
    # Временный холст для отрисовки только НОВОГО куска траекторий
    canvas = np.zeros(frame_shape, dtype=np.uint8)
    
    for ant_idx in range(n_tracks):
        traj = trajectories_chunk[:, ant_idx, :]
        
        # Фильтруем NaN. 
        # ВАЖНО: Если в исходном коде использовалось fill_value=np.nan, 
        # то тип данных float. Если вы перешли на int с sentinel, замените проверку.
        valid = ~np.any(np.isnan(traj), axis=1)
        pts = traj[valid]
        
        if len(pts) < 2:
            continue
            
        # Рисуем только новые сегменты на временном холсте
        # Используем color=1, чтобы потом сделать побитовое ИЛИ с основной маской
        cv.polylines(
            canvas,
            [np.ascontiguousarray(pts, dtype=np.int32)],
            isClosed=False,
            color=1,
            thickness=thickness
        )
    
    # Объединяем новое покрытие с накопленным.
    # Поскольку мы рисуем цветом 1, можно использовать побитовое ИЛИ или максимум.
    # Это гарантирует, что площадь не будет считаться дважды для одного и того же трека.
    np.bitwise_or(coverage_mask, canvas, out=coverage_mask)
    
    # Площадь SOI — это просто количество занятых пикселей
    return float(np.count_nonzero(coverage_mask))

In [ ]:
frame_shape = (800, 1300)
thikness = 10
coverage_mask = np.zeros(frame_shape, dtype=np.uint8)
start_frame = 0
result = []

df_test = pd.read_csv('../data/knn_kalman/pyeensis/S2190011.csv', dtype={'frame':np.uint16, 'track_id':np.uint16, 'x':np.float16, 'y':np.float16}, usecols=['frame', 'track_id', 'x', 'y'])
df_test = df_test[(df_test['x']>=0)&(df_test['y']>=0)]
df_test = df_test.round().astype({'x': np.uint16, 'y': np.uint16})

# df_test = df_test.round().astype(np.uint16)
# Убедитесь, что пороги отсортированы и уникальны
frame_thresholds = np.unique(
    np.exp(np.arange(np.log(10), np.log(df_test.frame.max()), 0.5)).astype(np.int32)
)


In [4]:
x_wide = df_test.pivot_table(columns='track_id', index='frame', values='x', fill_value=np.nan).astype(np.uint16)
y_wide = df_test.pivot_table(columns='track_id', index='frame', values='y', fill_value=np.nan).astype(np.uint16)
# new_trajectories = np.stack([x_wide.values, y_wide.values], axis=-1)

IntCastingNaNError: Cannot convert non-finite values (NA or inf) to integer.Replace or remove non-finite values or cast to an integer typethat supports these values (e.g. 'Int64')

In [15]:
display(df_test.__sizeof__()/1e6)
display(x_wide.__sizeof__()/1e6)
display(y_wide.__sizeof__()/1e6)

41.059584

NameError: name 'x_wide' is not defined

In [ ]:

for frame in frame_thresholds:
    # 1. Берем срез данных ДЛЯ НОВОГО ИНТЕРВАЛА [start_frame, frame)
    # Важно: фильтруем по значению колонки 'frame', а не по индексу строки!
    mask_new = (df_test['frame'] >= start_frame) & (df_test['frame'] < frame)
    df_slice_new = df_test[mask_new]
    
    if df_slice_new.empty:
        # Если в интервале нет данных, площадь не меняется, но порог всё равно фиксируем
        current_soi = float(np.count_nonzero(coverage_mask))
    else:
        x_wide = df_slice_new.pivot_table(columns='track_id', index='frame', values='x', fill_value=np.nan)
        y_wide = df_slice_new.pivot_table(columns='track_id', index='frame', values='y', fill_value=np.nan)
        new_trajectories = np.stack([x_wide.values, y_wide.values], axis=-1)
        
        # 2. Инкрементальное обновление маски
        current_soi = calc_soi_area_incremental(
            new_trajectories, 
            thikness, 
            frame_shape, 
            coverage_mask
        )
        
        del x_wide, y_wide, new_trajectories
        gc.collect()
    # 3. Расчёт L2 пути (также рекомендуется сделать инкрементальным, см. примечание ниже)
    # Здесь оставлена ваша оригинальная логика для корректности ответа на вопрос про SOI
    df_slice_full = df_test[df_test['frame'] < frame]
    x_wide_full = df_slice_full.pivot_table(columns='track_id', index='frame', values='x', fill_value=np.nan)
    y_wide_full = df_slice_full.pivot_table(columns='track_id', index='frame', values='y', fill_value=np.nan)
    traj_full = np.stack([x_wide_full.values, y_wide_full.values], axis=-1)
    
    diff = np.diff(traj_full, axis=0)
    total_l2_path = np.nansum(np.linalg.norm(diff, axis=2))
    del diff, x_wide_full, y_wide_full, traj_full
    gc.collect()
    result.append({
        'species': 'P.yeensis',
        'name': 'S2190011.csv',
        'frame_threshold': frame,
        # 'total_l1_path': total_l1_path,  # Убедитесь, что эта переменная определена
        'total_l2_path': total_l2_path,
        'soi_area': current_soi,
    })
    
    # 4. Сдвигаем окно
    start_frame = frame
    
    # Очистка памяти для нового среза
    gc.collect()